In [4]:
import pandas as pd

# Define the path to your local CSV file
filepath = 'Clustering_dataset.csv'

# Load the dataset into a pandas DataFrame
df = pd.read_csv(filepath)

# --- Good practice after loading ---
# Inspect the first few rows
print("--- Data Head ---")
print(df.head())

# Check for missing values and data types
print("\n--- Data Info ---")
df.info()

--- Data Head ---
   CustomerID  Gender   Age  Annual_Income  Spending_Score
0           1    Male  19.0           15.0            39.0
1           2    Male  21.0            NaN            81.0
2           3  Female  20.0           16.0             6.0
3           4  Female  23.0           16.0            77.0
4           5  Female  31.0           17.0            40.0

--- Data Info ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CustomerID      200 non-null    int64  
 1   Gender          194 non-null    object 
 2   Age             194 non-null    float64
 3   Annual_Income   194 non-null    float64
 4   Spending_Score  194 non-null    float64
dtypes: float64(3), int64(1), object(1)
memory usage: 7.9+ KB


In [6]:
# Before selecting columns, drop rows with any missing values
df.dropna(inplace=True)

# --- Good practice after loading ---
# Inspect the first few rows
print("--- Data Head ---")
print(df.head())

# Check for missing values and data types
print("\n--- Data Info ---")
df.info()

--- Data Head ---
   CustomerID  Gender   Age  Annual_Income  Spending_Score
0           1    Male  19.0           15.0            39.0
2           3  Female  20.0           16.0             6.0
3           4  Female  23.0           16.0            77.0
4           5  Female  31.0           17.0            40.0
5           6  Female  22.0           17.0            76.0

--- Data Info ---
<class 'pandas.core.frame.DataFrame'>
Index: 178 entries, 0 to 199
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   CustomerID      178 non-null    int64  
 1   Gender          178 non-null    object 
 2   Age             178 non-null    float64
 3   Annual_Income   178 non-null    float64
 4   Spending_Score  178 non-null    float64
dtypes: float64(3), int64(1), object(1)
memory usage: 8.3+ KB


In [7]:
from sklearn.preprocessing import StandardScaler

# 2. Select the best numerical features for clustering
features = df[['Age', 'Annual_Income', 'Spending_Score']]

# 3. Scale the selected features
scaler = StandardScaler()
scaled_features = scaler.fit_transform(features)

# The 'scaled_features' array is now ready for your K-Means algorithm
print("Data is scaled and ready for K-Means.")

Data is scaled and ready for K-Means.


In [8]:
import numpy as np

class MyKMeans:
    """
    A from-scratch implementation of the K-Means clustering algorithm.
    """
    def __init__(self, k, max_iters=100):
        """
        Initializes the MyKMeans instance.
        
        Parameters:
        - k (int): The number of clusters to form.
        - max_iters (int): Maximum number of iterations for the algorithm.
        """
        self.k = k
        self.max_iters = max_iters
        # To store the final cluster centroids
        self.centroids_ = None

    def _get_cluster_labels(self, X):
        """Helper function to assign each point to the nearest centroid."""
        # Calculate the distance from each point to each centroid
        # Broadcasting X (n_samples, n_features) and centroids (k, n_features)
        # results in a (n_samples, k, n_features) array.
        distances = np.sqrt(((X - self.centroids_[:, np.newaxis])**2).sum(axis=2))
        
        # Return the index of the closest centroid for each point
        return np.argmin(distances, axis=0)

    def fit(self, X):
        """
        Trains the K-Means model on the dataset X.
        
        Parameters:
        - X (np.array): Training data of shape (n_features, n_samples).
        """
        # 1. Initialize Centroids: Randomly select k data points
        n_features, n_samples = X.shape
        random_indices = np.random.choice(n_samples, self.k, replace=False)
        self.centroids_ = X[:, random_indices]

        # 2. Iterate and Refine
        for _ in range(self.max_iters):
            # Store old centroids to check for convergence
            old_centroids = self.centroids_.copy()
            
            # Assignment Step: Assign points to the closest centroid
            labels = self._get_cluster_labels(X)
            
            # Update Step: Calculate new centroids as the mean of assigned points
            new_centroids = np.array([X[:, labels == i].mean(axis=1) for i in range(self.k)]).T
            self.centroids_ = new_centroids
            
            # 3. Check for Convergence
            if np.all(old_centroids == self.centroids_):
                break
                
        return self

    def predict(self, X):
        """
        Assigns each data point in X to the nearest cluster.

        Parameters:
        - X (np.array): Data to predict on, shape (n_features, n_samples).
        
        Returns:
        - np.array: An array of cluster labels for each data point.
        """
        if self.centroids_ is None:
            raise RuntimeError("The model has not been fitted yet. Please call fit() first.")
        
        return self._get_cluster_labels(X)

    def getCost(self, X):
        """
        Calculates the Within-Cluster Sum of Squares (WCSS), also known as inertia.

        Parameters:
        - X (np.array): Data to calculate cost for, shape (n_features, n_samples).

        Returns:
        - float: The total WCSS cost.
        """
        cost = 0
        labels = self.predict(X)
        
        for i in range(self.k):
            # Get all points belonging to the current cluster
            cluster_points = X[:, labels == i]
            
            # If the cluster is not empty, calculate the squared distance
            if cluster_points.shape[1] > 0:
                centroid = self.centroids_[:, i].reshape(-1, 1)
                cost += np.sum((cluster_points - centroid)**2)
                
        return cost